# Sprint 2: Storage, Cleaning, and Preprocessing

This notebook builds a SQL metadata database for the ASL Alphabet dataset, validates image files, assigns train/validation/test splits, and prepares the dataset for preprocessing.

In [1]:
import os
import sqlite3
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

TRAIN_DIR = "../data/asl_alphabet_train/asl_alphabet_train"
TEST_DIR = "../data/asl_alphabet_test"
DB_PATH = "../asl_metadata.db"

print(os.path.exists(TRAIN_DIR))
print(os.listdir(TRAIN_DIR)[:10])

True
['.DS_Store', 'R', 'U', 'I', 'N', 'G', 'Z', 'T', 'S', 'A']


Dataset image files are stored locally in the ignored data/ folder, while the notebook stores relative image paths in SQLite so the workflow can be reproduced across team members’ machines.

## SQL Database Connection

SQLite is used to store image metadata, including relative file paths, labels, validation status, and split assignments.

In [2]:
conn = sqlite3.connect(DB_PATH)

## Build Image Metadata

This section scans the training image folders and creates one metadata record per image.
The database stores relative paths so the notebook can run on different machines.

In [3]:
records = []

for label in os.listdir(TRAIN_DIR):
    if label.startswith("."):
        continue

    class_path = os.path.join(TRAIN_DIR, label)

    if os.path.isdir(class_path):
        for image_file in os.listdir(class_path):
            if image_file.startswith("."):
                continue

            image_path = os.path.join(class_path, image_file)
            relative_path = os.path.relpath(image_path, TRAIN_DIR)

            records.append({
                "file_path": relative_path,
                "label": label
            })

df = pd.DataFrame(records)
df.head()

,file_path,label
0,R/R2837.jpg,R
1,R/R2189.jpg,R
2,R/R1480.jpg,R
3,R/R1494.jpg,R
4,R/R2823.jpg,R


## Image Cleaning and Validation

Each image is opened with PIL to confirm it can be loaded. The notebook records width, height, file format, validity status, and any issue notes.

In [4]:
widths = []
heights = []
formats = []
is_valids = []
issue_notes = []

for path in df["file_path"]:
    full_path = os.path.join(TRAIN_DIR, path)

    try:
        with Image.open(full_path) as img:
            widths.append(img.width)
            heights.append(img.height)
            formats.append(img.format)
            is_valids.append(1)
            issue_notes.append(None)
    except Exception as e:
        widths.append(None)
        heights.append(None)
        formats.append(None)
        is_valids.append(0)
        issue_notes.append(str(e))

df["width"] = widths
df["height"] = heights
df["file_format"] = formats
df["is_valid"] = is_valids
df["issue_notes"] = issue_notes

## Train, Validation, and Test Splits

The dataset is split using a stratified 70/15/15 split to preserve class balance across all classes.

In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

df["split"] = "unassigned"
df.loc[train_df.index, "split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"

## Saving Metadata to SQLite

The completed metadata table is saved to SQLite for later querying by the preprocessing pipeline.

In [6]:
df.to_sql("images", conn, if_exists="replace", index=False)

87000

## SQL Verification

These queries confirm that the database contains the expected image records, valid files, and balanced split assignments.

In [7]:
pd.read_sql_query("""
SELECT split, COUNT(*) AS count
FROM images
GROUP BY split
""", conn)

,split,count
0,test,13050
1,train,60900
2,val,13050


In [8]:
pd.read_sql_query("""
SELECT label, split, COUNT(*) AS count
FROM images
GROUP BY label, split
ORDER BY label, split
""", conn)

,label,split,count
0,A,test,450
1,A,train,2100
2,A,val,450
3,B,test,450
4,B,train,2100
...,...,...,...
82,nothing,train,2100
83,nothing,val,450
84,space,test,450
85,space,train,2100


In [9]:
pd.read_sql_query("""
SELECT is_valid, COUNT(*) AS count
FROM images
GROUP BY is_valid
""", conn)

,is_valid,count
0,1,87000


In [10]:
pd.read_sql_query("""
SELECT width, height, COUNT(*) AS count
FROM images
GROUP BY width, height
""", conn)

,width,height,count
0,200,200,87000


## Loading Split DataFrames from SQL

The train, validation, and test splits are loaded from SQL into Pandas DataFrames. These DataFrames will be passed into the preprocessing pipeline and PyTorch Dataset class.

In [11]:
train_df = pd.read_sql_query("""
SELECT file_path, label
FROM images
WHERE split = 'train'
AND is_valid = 1
""", conn)

val_df = pd.read_sql_query("""
SELECT file_path, label
FROM images
WHERE split = 'val'
AND is_valid = 1
""", conn)

test_df = pd.read_sql_query("""
SELECT file_path, label
FROM images
WHERE split = 'test'
AND is_valid = 1
""", conn)

train_df.shape, val_df.shape, test_df.shape

((60900, 2), (13050, 2), (13050, 2))

## Summary

This section created a SQLite metadata database for the ASL Alphabet dataset. Each image record includes a relative file path, class label, image dimensions, file format, validation status, and train/validation/test split assignment.

A stratified 70/15/15 split was used to preserve class balance. The database can now be queried to retrieve clean image records for preprocessing and model training.